# types_psutil Package Testing

This notebook tests the types_psutil package, which provides type stubs for psutil.

## Example 1: Import verification

In [ ]:
# types_psutil provides type stubs for psutil
# Let's import psutil and demonstrate type-safe system monitoring
import psutil
from typing import List, Dict, Optional, Tuple
import time

# Verify psutil is available with type information
print(f"psutil version: {psutil.__version__}")

# Get basic system information with type annotations
def get_system_info() -> Dict[str, str]:
    """Get basic system information with type safety."""
    boot_time: float = psutil.boot_time()
    boot_time_str: str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(boot_time))
    
    return {
        'platform': psutil.LINUX if hasattr(psutil, 'LINUX') else 'other',
        'boot_time': boot_time_str,
        'cpu_count_logical': str(psutil.cpu_count(logical=True)),
        'cpu_count_physical': str(psutil.cpu_count(logical=False)),
    }

system_info = get_system_info()

print("System Information (type-safe):")
for key, value in system_info.items():
    print(f"  {key}: {value}")

# Demonstrate CPU usage with proper typing
cpu_percent: float = psutil.cpu_percent(interval=1)
print(f"\nCurrent CPU usage: {cpu_percent}%")

## Example 2: Process information with type annotations

In [ ]:
# Demonstrate process management with type safety
import psutil
from typing import List, Dict, Any, Optional
import os

# Get current process information with type annotations
def get_current_process_info() -> Dict[str, Any]:
    """Get information about the current process with type safety."""
    current_process: psutil.Process = psutil.Process()
    
    return {
        'pid': current_process.pid,
        'name': current_process.name(),
        'status': current_process.status(),
        'create_time': current_process.create_time(),
        'cpu_percent': current_process.cpu_percent(),
        'memory_percent': current_process.memory_percent(),
        'num_threads': current_process.num_threads(),
    }

# Get list of running processes with type safety
def get_process_list(limit: int = 5) -> List[Dict[str, Any]]:
    """Get a list of running processes with type annotations."""
    processes: List[Dict[str, Any]] = []
    
    for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent']):
        try:
            proc_info: Dict[str, Any] = proc.info
            if proc_info['cpu_percent'] is not None:
                processes.append(proc_info)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    
    # Sort by CPU usage and return top processes
    processes.sort(key=lambda x: x['cpu_percent'] or 0, reverse=True)
    return processes[:limit]

# Display current process information
current_proc_info = get_current_process_info()
print("Current Process Information (type-safe):")
for key, value in current_proc_info.items():
    print(f"  {key}: {value}")

# Display top processes
print(f"\nTop 5 processes by CPU usage:")
top_processes = get_process_list(5)
for i, proc in enumerate(top_processes, 1):
    print(f"  {i}. PID {proc['pid']}: {proc['name']} - CPU: {proc['cpu_percent']}%")

## Example 3: Memory and CPU statistics with type annotations

In [ ]:
# Demonstrate memory monitoring with type safety
import psutil
from typing import NamedTuple, Dict, List

# Create type-safe memory information structure
class MemoryInfo(NamedTuple):
    total: int
    available: int
    used: int
    percent: float
    free: int

class SwapInfo(NamedTuple):
    total: int
    used: int
    free: int
    percent: float

# Get memory statistics with type safety
def get_memory_stats() -> Tuple[MemoryInfo, SwapInfo]:
    """Get system memory statistics with type annotations."""
    virtual_mem = psutil.virtual_memory()
    swap_mem = psutil.swap_memory()
    
    memory_info = MemoryInfo(
        total=virtual_mem.total,
        available=virtual_mem.available,
        used=virtual_mem.used,
        percent=virtual_mem.percent,
        free=virtual_mem.free
    )
    
    swap_info = SwapInfo(
        total=swap_mem.total,
        used=swap_mem.used,
        free=swap_mem.free,
        percent=swap_mem.percent
    )
    
    return memory_info, swap_info

# Get per-CPU statistics with type safety
def get_cpu_stats() -> Dict[str, Any]:
    """Get CPU statistics with type annotations."""
    cpu_times = psutil.cpu_times()
    cpu_percents: List[float] = psutil.cpu_percent(percpu=True, interval=0.1)
    
    return {
        'cpu_times_user': cpu_times.user,
        'cpu_times_system': cpu_times.system,
        'cpu_times_idle': cpu_times.idle,
        'per_cpu_usage': cpu_percents,
        'avg_cpu_usage': sum(cpu_percents) / len(cpu_percents),
        'cpu_count': len(cpu_percents)
    }

# Display memory information
memory_info, swap_info = get_memory_stats()
print("Memory Information (type-safe):")
print(f"  Total: {memory_info.total // (1024**3):.1f} GB")
print(f"  Used: {memory_info.used // (1024**3):.1f} GB ({memory_info.percent:.1f}%)")
print(f"  Available: {memory_info.available // (1024**3):.1f} GB")

print(f"\nSwap Information:")
print(f"  Total: {swap_info.total // (1024**3):.1f} GB")
print(f"  Used: {swap_info.used // (1024**3):.1f} GB ({swap_info.percent:.1f}%)")

# Display CPU information
cpu_stats = get_cpu_stats()
print(f"\nCPU Information:")
print(f"  CPU Count: {cpu_stats['cpu_count']}")
print(f"  Average Usage: {cpu_stats['avg_cpu_usage']:.1f}%")
print(f"  User Time: {cpu_stats['cpu_times_user']:.2f}s")
print(f"  System Time: {cpu_stats['cpu_times_system']:.2f}s")

## Example 4: Disk and network statistics with type annotations

In [ ]:
# Demonstrate disk and network monitoring with type safety
import psutil
from typing import List, Dict, NamedTuple, Optional

# Create type-safe structures for disk and network information
class DiskUsage(NamedTuple):
    device: str
    mountpoint: str
    total: int
    used: int
    free: int
    percent: float

class NetworkStats(NamedTuple):
    interface: str
    bytes_sent: int
    bytes_recv: int
    packets_sent: int
    packets_recv: int

# Get disk usage information with type safety
def get_disk_usage() -> List[DiskUsage]:
    """Get disk usage information for all mounted partitions."""
    disk_info: List[DiskUsage] = []
    
    for partition in psutil.disk_partitions():
        try:
            usage = psutil.disk_usage(partition.mountpoint)
            disk_info.append(DiskUsage(
                device=partition.device,
                mountpoint=partition.mountpoint,
                total=usage.total,
                used=usage.used,
                free=usage.free,
                percent=(usage.used / usage.total) * 100 if usage.total > 0 else 0
            ))
        except PermissionError:
            # Skip partitions we can't access
            continue
    
    return disk_info

# Get network statistics with type safety
def get_network_stats() -> List[NetworkStats]:
    """Get network interface statistics."""
    net_stats: List[NetworkStats] = []
    net_io = psutil.net_io_counters(pernic=True)
    
    for interface, stats in net_io.items():
        net_stats.append(NetworkStats(
            interface=interface,
            bytes_sent=stats.bytes_sent,
            bytes_recv=stats.bytes_recv,
            packets_sent=stats.packets_sent,
            packets_recv=stats.packets_recv
        ))
    
    return net_stats

# Display disk information
print("Disk Usage Information (type-safe):")
disk_usage = get_disk_usage()
for disk in disk_usage[:3]:  # Show first 3 disks
    print(f"  {disk.device} ({disk.mountpoint}):")
    print(f"    Total: {disk.total // (1024**3):.1f} GB")
    print(f"    Used: {disk.used // (1024**3):.1f} GB ({disk.percent:.1f}%)")
    print(f"    Free: {disk.free // (1024**3):.1f} GB")

# Display network information
print(f"\nNetwork Interface Statistics:")
network_stats = get_network_stats()
for net in network_stats[:3]:  # Show first 3 interfaces
    print(f"  {net.interface}:")
    print(f"    Bytes sent: {net.bytes_sent // (1024**2):.1f} MB")
    print(f"    Bytes received: {net.bytes_recv // (1024**2):.1f} MB")
    print(f"    Packets sent: {net.packets_sent:,}")

# Get system load average (on Unix systems)
try:
    load_avg: Tuple[float, float, float] = psutil.getloadavg()
    print(f"\nSystem Load Average: {load_avg[0]:.2f}, {load_avg[1]:.2f}, {load_avg[2]:.2f}")
except AttributeError:
    print(f"\nSystem Load Average: Not available on this platform")

## Example 5: A typed system monitoring class

In [ ]:
# Demonstrate advanced psutil features with type safety
import psutil
from typing import List, Dict, Optional, Callable, Any
import time

# Create a type-safe system monitoring class
class SystemMonitor:
    """Type-safe system monitoring class using psutil."""
    
    def __init__(self) -> None:
        self.start_time: float = time.time()
    
    def get_system_summary(self) -> Dict[str, Any]:
        """Get a comprehensive system summary with type safety."""
        return {
            'uptime_seconds': time.time() - psutil.boot_time(),
            'cpu_count': psutil.cpu_count(),
            'memory_total_gb': psutil.virtual_memory().total / (1024**3),
            'memory_used_percent': psutil.virtual_memory().percent,
            'disk_usage_percent': psutil.disk_usage('/').percent,
            'process_count': len(psutil.pids()),
            'users_logged_in': len(psutil.users()),
        }
    
    def monitor_process_by_name(self, process_name: str) -> Optional[Dict[str, Any]]:
        """Monitor a specific process by name with type safety."""
        for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_info']):
            try:
                if process_name.lower() in proc.info['name'].lower():
                    memory_info = proc.memory_info()
                    return {
                        'pid': proc.info['pid'],
                        'name': proc.info['name'],
                        'cpu_percent': proc.cpu_percent(),
                        'memory_mb': memory_info.rss / (1024**2),
                        'status': proc.status(),
                        'create_time': proc.create_time(),
                    }
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        return None
    
    def get_top_processes_by_memory(self, limit: int = 5) -> List[Dict[str, Any]]:
        """Get top processes by memory usage with type safety."""
        processes: List[Dict[str, Any]] = []
        
        for proc in psutil.process_iter(['pid', 'name', 'memory_percent']):
            try:
                proc_data = {
                    'pid': proc.info['pid'],
                    'name': proc.info['name'],
                    'memory_percent': proc.info['memory_percent'],
                    'memory_mb': proc.memory_info().rss / (1024**2),
                }
                processes.append(proc_data)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        
        # Sort by memory usage and return top processes
        processes.sort(key=lambda x: x['memory_percent'] or 0, reverse=True)
        return processes[:limit]

# Create and use the system monitor
monitor = SystemMonitor()

# Get system summary
summary = monitor.get_system_summary()
print("System Summary (type-safe monitoring):")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

# Monitor Python processes
python_proc = monitor.monitor_process_by_name("python")
if python_proc:
    print(f"\nPython Process Found:")
    print(f"  PID: {python_proc['pid']}")
    print(f"  Name: {python_proc['name']}")
    print(f"  Memory: {python_proc['memory_mb']:.1f} MB")
    print(f"  CPU: {python_proc['cpu_percent']:.1f}%")

# Get top memory consumers
print(f"\nTop 3 Processes by Memory Usage:")
top_procs = monitor.get_top_processes_by_memory(3)
for i, proc in enumerate(top_procs, 1):
    print(f"  {i}. {proc['name']} (PID {proc['pid']}): {proc['memory_mb']:.1f} MB ({proc['memory_percent']:.1f}%)")

print(f"\nTypes-psutil enables comprehensive type safety for:")
print("✓ System information gathering")
print("✓ Process monitoring and management") 
print("✓ Memory and CPU statistics")
print("✓ Disk and network monitoring")
print("✓ Advanced system monitoring classes")
print("✓ Error handling with proper exception types")